In [ ]:
##RUN THIS TO CONVERT LABELS INTO YOLO FORMAT

import os
import json
from pathlib import Path
from tqdm import tqdm

bdd100k_root = Path('100k')

class_names = ['car', 'truck', 'bus', 'motorcycle', 'bicycle', 'person', 'rider', 'traffic light', 'traffic sign', 'train']
class_to_idx = {name: idx for idx, name in enumerate(class_names)}

def convert_bdd100k_to_yolo(image_dir, img_width=1280, img_height=720):
    image_dir = Path(image_dir)
    labels_dir = image_dir.parent / image_dir.name / 'labels'
    labels_dir.mkdir(parents=True, exist_ok=True)
    
    json_files = list(image_dir.glob('*.json'))
    
    for json_path in tqdm(json_files, desc=f"Converting {image_dir.name}"):
        with open(json_path, 'r') as f:
            data = json.load(f)
        
        txt_name = json_path.stem + '.txt'
        txt_path = labels_dir / txt_name
        
        with open(txt_path, 'w') as txt_file:
            if 'frames' in data and len(data['frames']) > 0:
                for obj in data['frames'][0].get('objects', []):
                    if obj['category'] in class_to_idx and 'box2d' in obj:
                        box = obj['box2d']
                        x1, y1 = box['x1'], box['y1']
                        x2, y2 = box['x2'], box['y2']
                        
                        x_center = ((x1 + x2) / 2) / img_width
                        y_center = ((y1 + y2) / 2) / img_height
                        width = (x2 - x1) / img_width
                        height = (y2 - y1) / img_height
                        
                        class_id = class_to_idx[obj['category']]
                        txt_file.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

print("Converting BDD100K labels to YOLO format...")
convert_bdd100k_to_yolo(bdd100k_root / 'train')
convert_bdd100k_to_yolo(bdd100k_root / 'val')
print("Conversion complete!")